# Hands-On DBMS Interactive Laboratory Notebook
### Complete Step-by-Step Practical Labs: Architecture, CRUD, Joins, Normalization, Transactions & Indexing

Welcome to the **Database Management Systems (DBMS) Practical Lab**. This interactive notebook uses Python's built-in `sqlite3` engine and `pandas` so you can execute real SQL queries, test integrity constraints, normalize relations, simulate concurrent transactions, and inspect B+ Tree query plans directly inside Jupyter or Google Colab—with zero external server setup required.

---

## Lab Roadmap
1. **Environment Setup & Helper Utilities**
2. **Lab 1: Three-Schema Architecture, ER Mapping, Keys & DDL (`CREATE`, `ALTER`, `VIEW`)**
3. **Lab 2: Basic CRUD Operations (`INSERT`, `SELECT`, `UPDATE`, `DELETE`) & SQL Execution Order**
4. **Lab 3: Relational Algebra & SQL Joins (`INNER`, `LEFT`, Anti-Joins, `FULL OUTER`, Division)**
5. **Lab 4: Functional Dependencies, Armstrong's Axioms & Normalization (`1NF` $\rightarrow$ `BCNF`)**
6. **Lab 5: Transactions (`ACID`), Concurrency Control, Recovery & `B+ Tree` Indexing**

> **How to use this lab:**
> * Read each **Lab Task** specification carefully.
> * Fill in the `# TODO:` sections in the code cells.
> * Run the **Automated Grading / Verification Cell** below each task to check your work.
> * Expand the `<details><summary>Click to view Reference Solution</summary>` dropdown if you get stuck.


---
## 0. Environment Setup & SQL Helper Functions
Run the cell below to initialize an in-memory SQLite database with Foreign Key enforcement enabled (`PRAGMA foreign_keys = ON;`) and helper functions to display query results cleanly as Pandas DataFrames.


In [ ]:
import sqlite3
import pandas as pd
import time

# Connect to an in-memory SQLite database (or change ':memory:' to 'university_lab.db' to persist on disk)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# CRITICAL: Enable Foreign Key Referential Integrity enforcement in SQLite
cursor.execute("PRAGMA foreign_keys = ON;")

def run_query(sql, params=(), connection=conn):
    """Executes a SELECT query and returns a formatted Pandas DataFrame."""
    return pd.read_sql_query(sql, connection, params=params)

def exec_sql(sql, params=(), connection=conn):
    """Executes DDL/DML statements (CREATE, INSERT, UPDATE, DELETE) and prints affected rows."""
    cur = connection.cursor()
    cur.execute(sql, params)
    connection.commit()
    print(f"Statement executed successfully. Rows affected: {cur.rowcount}")
    return cur

def exec_script(sql_script, connection=conn):
    """Executes a multi-statement SQL script."""
    cur = connection.cursor()
    cur.executescript(sql_script)
    connection.commit()
    print("SQL script executed successfully.")

print("DBMS Lab Environment Initialized Successfully!")


---
## Lab 1: Architecture, ER Modeling, Keys & DDL
### Concepts Covered
* **Three-Schema Architecture:**
  * **Internal Level:** Physical storage & indexes.
  * **Conceptual Level:** Logical tables, columns, and constraints (`CREATE TABLE`).
  * **External Level:** User-specific views (`CREATE VIEW`) providing **Logical Data Independence**.
* **Database Keys & Constraints:**
  * **Primary Key (PK):** Unique + `NOT NULL` (Entity Integrity).
  * **Alternate Key (AK):** Candidate key not chosen as PK, enforced via `UNIQUE`.
  * **Foreign Key (FK):** Enforces Referential Integrity across $1:N$ and $M:N$ relationships.
  * **Composite Key:** Primary key formed by combining multiple attributes (used in $M:N$ bridge tables).

---
### Task 1.1: Create Strong Entities & $1:N$ Relationships (`DDL`)
Write DDL statements to create the following tables:
1. `Departments`:
   * `dept_id`: `INTEGER PRIMARY KEY` (Surrogate Key)
   * `dept_code`: `TEXT NOT NULL UNIQUE` (Alternate Key, e.g., `'CS'`, `'EE'`)
   * `dept_name`: `TEXT NOT NULL UNIQUE`
   * `building`: `TEXT NOT NULL`
2. `Instructors`:
   * `instructor_id`: `INTEGER PRIMARY KEY`
   * `full_name`: `TEXT NOT NULL`
   * `email`: `TEXT NOT NULL UNIQUE` (Alternate Key)
   * `salary`: `REAL NOT NULL CHECK (salary > 0)`
   * `dept_id`: `INTEGER`, Foreign Key referencing `Departments(dept_id)` with `ON DELETE SET NULL`


In [ ]:
# TODO: Write and execute the CREATE TABLE statements for Departments and Instructors

ddl_task_1_1 = """
DROP TABLE IF EXISTS Enrollments;
DROP TABLE IF EXISTS Courses;
DROP TABLE IF EXISTS Students;
DROP TABLE IF EXISTS Instructors;
DROP TABLE IF EXISTS Departments;

CREATE TABLE Departments (
    dept_id INTEGER PRIMARY KEY,
    dept_code TEXT NOT NULL UNIQUE,
    dept_name TEXT NOT NULL UNIQUE,
    building TEXT NOT NULL
);

CREATE TABLE Instructors (
    instructor_id INTEGER PRIMARY KEY,
    full_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    salary REAL NOT NULL CHECK (salary > 0),
    dept_id INTEGER,
    FOREIGN KEY (dept_id) REFERENCES Departments(dept_id) ON DELETE SET NULL
);
"""

exec_script(ddl_task_1_1)


### Task 1.2: Create $M:N$ Bridge Table with a Composite Primary Key
In an ER model, a many-to-many ($M:N$) relationship between `Students` and `Courses` is reduced to a **bridge/junction table** (`Enrollments`) whose Primary Key is the composite of both Foreign Keys.

Create the following three tables:
1. `Students`:
   * `student_id`: `INTEGER PRIMARY KEY`
   * `roll_no`: `TEXT NOT NULL UNIQUE` (Alternate Key)
   * `student_name`: `TEXT NOT NULL`
   * `major_dept_id`: `INTEGER`, Foreign Key referencing `Departments(dept_id)`
   * `gpa`: `REAL DEFAULT 0.0 CHECK (gpa >= 0.0 AND gpa <= 4.0)`
2. `Courses`:
   * `course_id`: `INTEGER PRIMARY KEY`
   * `course_code`: `TEXT NOT NULL UNIQUE`
   * `title`: `TEXT NOT NULL`
   * `credits`: `INTEGER NOT NULL CHECK (credits BETWEEN 1 AND 6)`
   * `dept_id`: `INTEGER NOT NULL`, Foreign Key referencing `Departments(dept_id)`
3. `Enrollments` ($M:N$ Bridge Table):
   * `student_id`: `INTEGER NOT NULL`
   * `course_id`: `INTEGER NOT NULL`
   * `semester`: `TEXT NOT NULL` (e.g., `'Fall-2026'`)
   * `grade`: `TEXT CHECK (grade IN ('A', 'B', 'C', 'D', 'F', NULL))`
   * **Composite Primary Key:** `(student_id, course_id)`
   * **Foreign Keys:** Reference `Students(student_id) ON DELETE CASCADE` and `Courses(course_id) ON DELETE CASCADE`


In [ ]:
# TODO: Create Students, Courses, and the M:N Bridge Table Enrollments

ddl_task_1_2 = """
CREATE TABLE Students (
    student_id INTEGER PRIMARY KEY,
    roll_no TEXT NOT NULL UNIQUE,
    student_name TEXT NOT NULL,
    major_dept_id INTEGER,
    gpa REAL DEFAULT 0.0 CHECK (gpa >= 0.0 AND gpa <= 4.0),
    FOREIGN KEY (major_dept_id) REFERENCES Departments(dept_id)
);

CREATE TABLE Courses (
    course_id INTEGER PRIMARY KEY,
    course_code TEXT NOT NULL UNIQUE,
    title TEXT NOT NULL,
    credits INTEGER NOT NULL CHECK (credits BETWEEN 1 AND 6),
    dept_id INTEGER NOT NULL,
    FOREIGN KEY (dept_id) REFERENCES Departments(dept_id)
);

CREATE TABLE Enrollments (
    student_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    semester TEXT NOT NULL,
    grade TEXT CHECK (grade IN ('A', 'B', 'C', 'D', 'F')),
    PRIMARY KEY (student_id, course_id),
    FOREIGN KEY (student_id) REFERENCES Students(student_id) ON DELETE CASCADE,
    FOREIGN KEY (course_id) REFERENCES Courses(course_id) ON DELETE CASCADE
);
"""

exec_script(ddl_task_1_2)


### Task 1.3: Logical Data Independence & External Schema (`ALTER TABLE` & `CREATE VIEW`)
1. Create an **External View** named `Public_Instructor_Directory` that exposes `instructor_id`, `full_name`, `email`, and `dept_id` from `Instructors`, hiding the sensitive `salary` column.
2. Modify the **Conceptual Schema** by adding an `office_room TEXT DEFAULT 'TBD'` column to `Instructors` using `ALTER TABLE`.
3. Verify **Logical Data Independence**: query `Public_Instructor_Directory` to confirm that the external view continues to work seamlessly despite the conceptual table change.


In [ ]:
# TODO: Create the view Public_Instructor_Directory and alter the Instructors table

ddl_task_1_3 = """
DROP VIEW IF EXISTS Public_Instructor_Directory;

CREATE VIEW Public_Instructor_Directory AS
SELECT instructor_id, full_name, email, dept_id
FROM Instructors;

ALTER TABLE Instructors ADD COLUMN office_room TEXT DEFAULT 'TBD';
"""

exec_script(ddl_task_1_3)

# Verify Lab 1 Schema & View
tables = run_query("SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY type, name;")
display(tables)
assert len(tables) == 6, "Expected 5 tables and 1 view!"
print("Lab 1 Verification Passed!")


---
## Lab 2: Basic CRUD Operations (DML) & SQL Execution Order
### Concepts Covered
* **CRUD Mapping to SQL DML:**
  * **C**reate $\rightarrow$ `INSERT INTO`
  * **R**ead $\rightarrow$ `SELECT`
  * **U**pdate $\rightarrow$ `UPDATE ... SET ... WHERE`
  * **D**elete $\rightarrow$ `DELETE FROM ... WHERE`
* **SQL Execution Order:**
  `FROM / JOIN` $\rightarrow$ `WHERE` $\rightarrow$ `GROUP BY` $\rightarrow$ `HAVING` $\rightarrow$ `SELECT` $\rightarrow$ `DISTINCT` $\rightarrow$ `ORDER BY` $\rightarrow$ `LIMIT / OFFSET`
* **`DELETE` (DML) vs. `TRUNCATE` (DDL) vs. `DROP` (DDL)**

---
### Task 2.1: CREATE (`INSERT INTO`) & Constraint Testing
1. Run the cell below to populate sample records across `Departments`, `Instructors`, `Courses`, `Students`, and `Enrollments`.
2. Write a test block that attempts to insert a duplicate `email` (Alternate Key violation) and a negative `salary` (`CHECK` constraint violation), catching the `sqlite3.IntegrityError`.


In [ ]:
# Seed the database with initial records (CREATE / INSERT)
seed_data_sql = """
DELETE FROM Enrollments;
DELETE FROM Courses;
DELETE FROM Students;
DELETE FROM Instructors;
DELETE FROM Departments;

INSERT INTO Departments (dept_id, dept_code, dept_name, building) VALUES
(1, 'CS', 'Computer Science', 'Turing Hall'),
(2, 'EE', 'Electrical Engineering', 'Tesla Lab'),
(3, 'ME', 'Mechanical Engineering', 'Newton Block'),
(4, 'PH', 'Physics', 'Feynman Center');

INSERT INTO Instructors (instructor_id, full_name, email, salary, dept_id, office_room) VALUES
(101, 'Dr. Ada Lovelace', 'ada@univ.edu', 125000, 1, 'TH-301'),
(102, 'Dr. Alan Turing', 'turing@univ.edu', 135000, 1, 'TH-305'),
(103, 'Dr. Grace Hopper', 'hopper@univ.edu', 128000, 1, 'TH-210'),
(104, 'Dr. Nikola Tesla', 'tesla@univ.edu', 118000, 2, 'TL-101'),
(105, 'Dr. Claude Shannon', 'shannon@univ.edu', 122000, 2, 'TL-204'),
(106, 'Dr. Richard Feynman', 'feynman@univ.edu', 130000, 4, 'FC-401');

INSERT INTO Courses (course_id, course_code, title, credits, dept_id) VALUES
(501, 'CS101', 'Database Management Systems', 4, 1),
(502, 'CS102', 'Operating Systems', 4, 1),
(503, 'CS103', 'Distributed Systems', 3, 1),
(504, 'EE201', 'Digital Signal Processing', 4, 2),
(505, 'PH301', 'Quantum Mechanics', 3, 4);

INSERT INTO Students (student_id, roll_no, student_name, major_dept_id, gpa) VALUES
(1001, 'R2026-01', 'Aarav Sharma', 1, 3.9),
(1002, 'R2026-02', 'Diya Patel', 1, 3.7),
(1003, 'R2026-03', 'Rohan Verma', 2, 3.4),
(1004, 'R2026-04', 'Meera Nair', 1, 3.95),
(1005, 'R2026-05', 'Kabir Singh', 3, 3.1),
(1006, 'R2026-06', 'Ananya Rao', NULL, 3.6);

INSERT INTO Enrollments (student_id, course_id, semester, grade) VALUES
(1001, 501, 'Fall-2026', 'A'),
(1001, 502, 'Fall-2026', 'A'),
(1001, 503, 'Fall-2026', 'B'),
(1002, 501, 'Fall-2026', 'A'),
(1002, 502, 'Fall-2026', 'B'),
(1003, 504, 'Fall-2026', 'A'),
(1004, 501, 'Fall-2026', 'A'),
(1004, 502, 'Fall-2026', 'A'),
(1004, 503, 'Fall-2026', 'A'),
(1004, 504, 'Fall-2026', 'B');
"""

exec_script(seed_data_sql)

# Verify Integrity Constraints catch invalid inserts
try:
    exec_sql("INSERT INTO Instructors (instructor_id, full_name, email, salary, dept_id) VALUES (999, 'Clone', 'ada@univ.edu', 90000, 1);")
except sqlite3.IntegrityError as e:
    print(f"[Expected Constraint Block] Alternate Key UNIQUE violation caught: {e}")

try:
    exec_sql("INSERT INTO Enrollments (student_id, course_id, semester, grade) VALUES (1001, 9999, 'Fall-2026', 'A');")
except sqlite3.IntegrityError as e:
    print(f"[Expected Constraint Block] Foreign Key Referential Integrity violation caught: {e}")


### Task 2.2: READ (`SELECT`) & SQL Execution Order Pipeline
Write a single SQL query that demonstrates all 8 stages of the SQL execution pipeline:
1. **`FROM / JOIN`**: Join `Departments d` and `Instructors i` on `d.dept_id = i.dept_id`.
2. **`WHERE`**: Filter only instructors whose `salary >= 120000`.
3. **`GROUP BY`**: Group by `d.dept_code`, `d.dept_name`.
4. **`HAVING`**: Keep only departments having `COUNT(i.instructor_id) >= 1` and `AVG(i.salary) > 121000`.
5. **`SELECT`**: Output `d.dept_code`, `d.dept_name`, `COUNT(i.instructor_id) AS high_earner_count`, and `ROUND(AVG(i.salary), 2) AS avg_salary`.
6. **`ORDER BY`**: Order by `avg_salary DESC`.
7. **`LIMIT`**: Limit to the top `2` results.


In [ ]:
# TODO: Write the SELECT query following the specification in Task 2.2

query_task_2_2 = """
SELECT 
    d.dept_code,
    d.dept_name,
    COUNT(i.instructor_id) AS high_earner_count,
    ROUND(AVG(i.salary), 2) AS avg_salary
FROM Departments d
JOIN Instructors i ON d.dept_id = i.dept_id
WHERE i.salary >= 120000
GROUP BY d.dept_code, d.dept_name
HAVING COUNT(i.instructor_id) >= 1 AND AVG(i.salary) > 121000
ORDER BY avg_salary DESC
LIMIT 2;
"""

df_2_2 = run_query(query_task_2_2)
display(df_2_2)
assert len(df_2_2) == 2 and df_2_2.iloc[0]["dept_code"] == "PH", "Check your GROUP BY, HAVING, or ORDER BY clause!"
print("Task 2.2 Verification Passed!")


### Task 2.3: UPDATE & DELETE Operations (with Cascade Verification)
1. **UPDATE**: Give a **5% salary raise** (`salary = salary * 1.05`) to all instructors in the Computer Science department (`dept_id = 1`).
2. **DELETE & ON DELETE CASCADE**:
   * Insert a temporary student `student_id = 1099` and enroll them in course `501`.
   * Delete student `1099` from `Students`.
   * Verify that their enrollment row in `Enrollments` was automatically removed via `ON DELETE CASCADE`.


In [ ]:
# 1. UPDATE Computer Science instructors' salaries by +5%
exec_sql("UPDATE Instructors SET salary = ROUND(salary * 1.05, 2) WHERE dept_id = 1;")
display(run_query("SELECT instructor_id, full_name, salary FROM Instructors WHERE dept_id = 1;"))

# 2. Test DELETE with ON DELETE CASCADE
exec_sql("INSERT INTO Students (student_id, roll_no, student_name, major_dept_id, gpa) VALUES (1099, 'R2026-99', 'Temp Student', 1, 3.0);")
exec_sql("INSERT INTO Enrollments (student_id, course_id, semester, grade) VALUES (1099, 501, 'Fall-2026', 'B');")

print("Enrollments for student 1099 before DELETE:", len(run_query("SELECT * FROM Enrollments WHERE student_id = 1099;")))

# Delete parent row in Students
exec_sql("DELETE FROM Students WHERE student_id = 1099;")

remaining = len(run_query("SELECT * FROM Enrollments WHERE student_id = 1099;"))
print("Enrollments for student 1099 after DELETE:", remaining)
assert remaining == 0, "ON DELETE CASCADE should have deleted the child enrollment row!"
print("Task 2.3 CRUD Verification Passed!")


---
## Lab 3: Relational Algebra & Complete SQL Joins Reference
### Concepts Covered
* **Relational Algebra Operators:**
  * Selection ($\sigma$), Projection ($\pi$), Rename ($\rho$), Set Union ($\cup$), Set Difference ($-$), Cartesian Product ($\times$), Natural Join ($\bowtie$), Division ($\div$).
* **SQL Joins Set Operations:**
  * `INNER JOIN`: Matching rows in both $A$ and $B$.
  * `LEFT JOIN`: All rows from $A$ + matched rows from $B$.
  * `LEFT EXCLUDING JOIN` (Left Anti-Join): `LEFT JOIN ... WHERE B.key IS NULL` (rows in $A$ with no match in $B$).
  * `FULL OUTER JOIN`: All rows from both $A$ and $B$.
  * `FULL EXCLUDING JOIN`: Rows unique to $A$ or unique to $B$ (`WHERE A.key IS NULL OR B.key IS NULL`).

---
### Task 3.1: Left Join vs. Left Excluding Join (Anti-Join)
1. Write a **Left Excluding Join** query (`query_unassigned_depts`) to find all `Departments` that currently have **no instructors assigned** (`i.instructor_id IS NULL`).
2. Write a **Left Excluding Join** query (`query_unenrolled_students`) to find all `Students` who have **zero course enrollments** in `Enrollments`.


In [ ]:
# TODO: Write Left Anti-Join queries

query_unassigned_depts = """
SELECT d.dept_id, d.dept_code, d.dept_name
FROM Departments d
LEFT JOIN Instructors i ON d.dept_id = i.dept_id
WHERE i.instructor_id IS NULL;
"""

query_unenrolled_students = """
SELECT s.student_id, s.roll_no, s.student_name
FROM Students s
LEFT JOIN Enrollments e ON s.student_id = e.student_id
WHERE e.course_id IS NULL;
"""

print("Departments with no instructors:")
df_unassigned = run_query(query_unassigned_depts)
display(df_unassigned)

print("Students with no course enrollments:")
df_unenrolled = run_query(query_unenrolled_students)
display(df_unenrolled)

assert list(df_unassigned["dept_code"]) == ["ME"], "Mechanical Engineering (ME) should have no instructors!"
assert set(df_unenrolled["student_id"]) == {1005, 1006}, "Students 1005 and 1006 have no enrollments!"
print("Task 3.1 Verification Passed!")


### Task 3.2: Full Outer Join & Full Excluding Join
Notice that student `1006` (`Ananya Rao`) has `major_dept_id = NULL` (undeclared major), and Department `3` (`Mechanical Engineering`, if we check instructors) has no instructors.
Write a **Full Excluding Join** between `Departments d` and `Instructors i` (`FULL OUTER JOIN ... WHERE d.dept_id IS NULL OR i.dept_id IS NULL`) to list either departments without instructors OR instructors without a department.


In [ ]:
# Insert a visiting instructor with no department (dept_id = NULL) to test Full Outer Join
exec_sql("INSERT OR IGNORE INTO Instructors (instructor_id, full_name, email, salary, dept_id) VALUES (107, 'Dr. John von Neumann', 'neumann@univ.edu', 140000, NULL);")

query_full_excluding = """
SELECT d.dept_code, d.dept_name, i.instructor_id, i.full_name
FROM Departments d
FULL OUTER JOIN Instructors i ON d.dept_id = i.dept_id
WHERE d.dept_id IS NULL OR i.dept_id IS NULL;
"""

df_full_excl = run_query(query_full_excluding)
display(df_full_excl)
assert len(df_full_excl) == 2, "Should return 1 unmatched department (ME) and 1 unmatched instructor (von Neumann)!"
print("Task 3.2 Verification Passed!")


### Task 3.3: Relational Division ($R \div S$) — "For All" Queries
In Relational Algebra, **Division ($R \div S$)** finds entities in $R$ associated with **every** tuple in $S$.
**Problem:** Find the `student_id` and `student_name` of students who are enrolled in **ALL** Computer Science courses (`dept_id = 1`, i.e., `CS101`, `CS102`, and `CS103`).

*Hint:* You can implement Relational Division in SQL either using:
* **Method A (Double `NOT EXISTS`):** "There does not exist a CS course that this student has not enrolled in."
* **Method B (`GROUP BY` + `HAVING COUNT`):** Count distinct CS courses taken by the student and match against `(SELECT COUNT(*) FROM Courses WHERE dept_id = 1)`.


In [ ]:
# TODO: Write a Relational Division query to find students enrolled in ALL Computer Science courses (dept_id = 1)

query_relational_division = """
SELECT s.student_id, s.roll_no, s.student_name
FROM Students s
WHERE NOT EXISTS (
    SELECT c.course_id
    FROM Courses c
    WHERE c.dept_id = 1
    EXCEPT
    SELECT e.course_id
    FROM Enrollments e
    WHERE e.student_id = s.student_id
);
"""

df_division = run_query(query_relational_division)
display(df_division)
assert set(df_division["student_id"]) == {1001, 1004}, "Students 1001 (Aarav) and 1004 (Meera) took all 3 CS courses!"
print("Task 3.3 Relational Division Verification Passed!")


---
## Lab 4: Functional Dependencies, Armstrong's Axioms & Normalization (`1NF` $\rightarrow$ `BCNF`)
### Concepts Covered
* **Armstrong's Axioms:** Reflexivity, Augmentation, Transitivity (plus Union & Decomposition).
* **Attribute Closure ($X^+$):** Set of all attributes functionally determined by $X$. If $X^+$ contains all attributes of $R$, $X$ is a **Super Key**.
* **Normal Forms Hierarchy:**
  * **1NF:** Atomic values only.
  * **2NF:** 1NF + No **Partial Dependency** (no non-prime attribute depends on a proper subset of a composite Candidate Key).
  * **3NF:** 2NF + No **Transitive Dependency** (for every non-trivial $X \rightarrow A$, $X$ is a Super Key or $A$ is prime).
  * **BCNF:** For every non-trivial $X \rightarrow A$, $X$ **must** be a Super Key.
* **Lossless-Join Decomposition:** Decomposing $R$ into $R_1$ and $R_2$ is lossless iff $(R_1 \cap R_2) \rightarrow R_1$ or $(R_1 \cap R_2) \rightarrow R_2$.

---
### Task 4.1: Compute Attribute Closure ($X^+$) & Candidate Keys in Python
Complete/run the function `compute_closure(attributes, fds)` that repeatedly applies Armstrong's **Transitivity/Union** rule until no new attributes can be added, and use it to verify the Candidate Key of a denormalized relation `R(student_id, course_id, student_name, dept_code, dept_building, instructor_id, grade)` with FDs:
* $\{student\_id, course\_id\} \rightarrow \{grade\}$
* $\{student\_id\} \rightarrow \{student\_name, dept\_code\}$
* $\{dept\_code\} \rightarrow \{dept\_building\}$
* $\{course\_id\} \rightarrow \{instructor\_id\}$


In [ ]:
from itertools import combinations

R_attrs = {"student_id", "course_id", "student_name", "dept_code", "dept_building", "instructor_id", "grade"}

# Functional Dependencies represented as pairs: (set_of_determinants, set_of_dependents)
FDs = [
    ({"student_id", "course_id"}, {"grade"}),
    ({"student_id"}, {"student_name", "dept_code"}),
    ({"dept_code"}, {"dept_building"}),
    ({"course_id"}, {"instructor_id"})
]

def compute_closure(seed_attrs, fds):
    """Computes the attribute closure X+ under a set of Functional Dependencies."""
    closure = set(seed_attrs)
    changed = True
    while changed:
        changed = False
        for lhs, rhs in fds:
            if lhs.issubset(closure) and not rhs.issubset(closure):
                closure.update(rhs)
                changed = True
    return closure

def find_candidate_keys(all_attrs, fds):
    """Finds all minimal Super Keys (Candidate Keys) of relation R."""
    candidate_keys = []
    for r in range(1, len(all_attrs) + 1):
        for subset in combinations(all_attrs, r):
            subset_set = set(subset)
            # Check if any already-found CK is a subset (minimality check)
            if any(ck.issubset(subset_set) for ck in candidate_keys):
                continue
            if compute_closure(subset_set, fds) == all_attrs:
                candidate_keys.append(subset_set)
    return candidate_keys

ck_list = find_candidate_keys(R_attrs, FDs)
print("Closure of {'student_id'}:", compute_closure({"student_id"}, FDs))
print("Closure of {'student_id', 'course_id'}:", compute_closure({"student_id", "course_id"}, FDs))
print("Discovered Candidate Key(s):", ck_list)
assert ck_list == [{"student_id", "course_id"}], "The sole Candidate Key should be {student_id, course_id}!"
print("Task 4.1 Verification Passed!")


### Task 4.2: Lossless-Join Decomposition into BCNF via SQL
In Task 4.1, our relation `Denorm_Records(student_id, course_id, student_name, dept_code, dept_building, instructor_id, grade)` has Candidate Key `(student_id, course_id)`.
Notice how it violates **2NF** and **3NF/BCNF**:
1. **Partial Dependencies (violates 2NF):**
   * `student_id -> student_name, dept_code` (depends only on part of the composite key!)
   * `course_id -> instructor_id` (depends only on part of the composite key!)
2. **Transitive Dependency (violates 3NF/BCNF):**
   * `student_id -> dept_code -> dept_building` (`dept_building` depends on non-key `dept_code`!)

Run the cell below to decompose `Denorm_Records` into 4 **BCNF** tables (`BCNF_Departments`, `BCNF_Students`, `BCNF_Courses`, `BCNF_Enrollments`) and verify that joining them back produces the **exact same 6 rows** (zero spurious tuples = **Lossless-Join Decomposition**).


In [ ]:
# 1. Create and populate the unnormalized/denormalized table
denorm_setup_sql = """
DROP TABLE IF EXISTS Denorm_Records;
CREATE TABLE Denorm_Records (
    student_id INTEGER,
    course_id INTEGER,
    student_name TEXT,
    dept_code TEXT,
    dept_building TEXT,
    instructor_id INTEGER,
    grade TEXT
);

INSERT INTO Denorm_Records VALUES
(1001, 501, 'Aarav Sharma', 'CS', 'Turing Hall', 101, 'A'),
(1001, 502, 'Aarav Sharma', 'CS', 'Turing Hall', 102, 'A'),
(1002, 501, 'Diya Patel', 'CS', 'Turing Hall', 101, 'B'),
(1003, 504, 'Rohan Verma', 'EE', 'Tesla Lab', 104, 'A'),
(1004, 501, 'Meera Nair', 'CS', 'Turing Hall', 101, 'A'),
(1004, 504, 'Meera Nair', 'CS', 'Turing Hall', 104, 'B');
"""
exec_script(denorm_setup_sql)

# 2. Decompose into BCNF tables using SQL projection (SELECT DISTINCT)
bcnf_decomposition_sql = """
DROP TABLE IF EXISTS BCNF_Departments;
DROP TABLE IF EXISTS BCNF_Students;
DROP TABLE IF EXISTS BCNF_Courses;
DROP TABLE IF EXISTS BCNF_Enrollments;

-- R1: dept_code -> dept_building
CREATE TABLE BCNF_Departments AS
SELECT DISTINCT dept_code, dept_building FROM Denorm_Records;

-- R2: student_id -> student_name, dept_code
CREATE TABLE BCNF_Students AS
SELECT DISTINCT student_id, student_name, dept_code FROM Denorm_Records;

-- R3: course_id -> instructor_id
CREATE TABLE BCNF_Courses AS
SELECT DISTINCT course_id, instructor_id FROM Denorm_Records;

-- R4: (student_id, course_id) -> grade
CREATE TABLE BCNF_Enrollments AS
SELECT DISTINCT student_id, course_id, grade FROM Denorm_Records;
"""
exec_script(bcnf_decomposition_sql)

# 3. Verify Lossless-Join Property: Natural Join of R1, R2, R3, R4 must equal original Denorm_Records
reconstructed_df = run_query("""
SELECT e.student_id, e.course_id, s.student_name, s.dept_code, d.dept_building, c.instructor_id, e.grade
FROM BCNF_Enrollments e
JOIN BCNF_Students s ON e.student_id = s.student_id
JOIN BCNF_Departments d ON s.dept_code = d.dept_code
JOIN BCNF_Courses c ON e.course_id = c.course_id
ORDER BY e.student_id, e.course_id;
""")

original_df = run_query("SELECT * FROM Denorm_Records ORDER BY student_id, course_id;")
display(reconstructed_df)
assert reconstructed_df.equals(original_df), "Lossless join failed: reconstructed relation differs from original!"
print("Task 4.2 BCNF Lossless-Join Decomposition Verified (0 Spurious Tuples)!")


---
## Lab 5: Transactions (`ACID`), TCL (`SAVEPOINT`, `ROLLBACK`), & `B+ Tree` Indexing
### Concepts Covered
* **ACID Properties:**
  * **Atomicity:** All-or-nothing execution (`UNDO` log).
  * **Consistency:** Preserves database invariants (e.g., total money across accounts remains constant).
  * **Isolation:** Concurrent transactions do not interfere (`Locks` / `MVCC`).
  * **Durability:** Committed changes survive crashes (`Write-Ahead Logging` / `REDO`).
* **TCL Commands:** `BEGIN TRANSACTION`, `SAVEPOINT`, `ROLLBACK TO SAVEPOINT`, `COMMIT`.
* **Indexing (`B+ Tree`):**
  * Without an index, filtering on a non-key column requires a **Full Table Scan** ($\mathcal{O}(N)$).
  * Creating a `B+ Tree` index (`CREATE INDEX`) enables logarithmic tree traversal ($\mathcal{O}(\log_B N)$) and fast linked-leaf range scans.

---
### Task 5.1: Atomic Scholarship Transfer with `SAVEPOINT` & `ROLLBACK`
Simulate a university scholarship transfer where:
1. We deduct `$5,000` from `University_Grant_Pool` (`account_id = 1`) and credit `$5,000` to `Student_Wallet` (`account_id = 2`).
2. We create a `SAVEPOINT after_first_transfer;`.
3. Next, we attempt a second transfer of `$50,000` that violates the `CHECK (balance >= 0)` constraint!
4. We catch the error, execute `ROLLBACK TO SAVEPOINT after_first_transfer;`, and `COMMIT;` so the first valid transfer succeeds atomically while the invalid second transfer is undone.


In [ ]:
# Create Accounts table with a non-negative balance CHECK constraint
exec_script("""
DROP TABLE IF EXISTS Accounts;
CREATE TABLE Accounts (
    account_id INTEGER PRIMARY KEY,
    account_name TEXT NOT NULL,
    balance REAL NOT NULL CHECK (balance >= 0)
);
INSERT INTO Accounts VALUES (1, 'University_Grant_Pool', 20000.0);
INSERT INTO Accounts VALUES (2, 'Aarav_Scholarship_Wallet', 1000.0);
""")

# Manage transactions manually for TCL demonstration
cur = conn.cursor()
cur.execute("BEGIN;")

# Step 1: Valid $5,000 transfer
cur.execute("UPDATE Accounts SET balance = balance - 5000 WHERE account_id = 1;")
cur.execute("UPDATE Accounts SET balance = balance + 5000 WHERE account_id = 2;")
cur.execute("SAVEPOINT after_first_transfer;")

# Step 2: Invalid $50,000 transfer (exceeds remaining $15,000 pool balance)
try:
    cur.execute("UPDATE Accounts SET balance = balance - 50000 WHERE account_id = 1;")
    cur.execute("UPDATE Accounts SET balance = balance + 50000 WHERE account_id = 2;")
except sqlite3.IntegrityError as e:
    print(f"[Transaction Alert] Second transfer failed ({e}). Rolling back to SAVEPOINT 'after_first_transfer'...")
    cur.execute("ROLLBACK TO SAVEPOINT after_first_transfer;")

cur.execute("COMMIT;")

df_accounts = run_query("SELECT * FROM Accounts;")
display(df_accounts)
assert list(df_accounts["balance"]) == [15000.0, 6000.0], "Balances should be 15000 and 6000 after partial rollback!"
print("Task 5.1 ACID & SAVEPOINT Verification Passed!")


### Task 5.2: Precedence (Serialization) Graph Cycle Detector for Conflict Serializability
Two operations conflict if they are in **different transactions**, operate on the **same data item**, and **at least one is a Write** ($R_i(X)\text{--}W_j(X)$, $W_i(X)\text{--}R_j(X)$, or $W_i(X)\text{--}W_j(X)$).
A schedule is **Conflict Serializable** if and only if its **Precedence Graph has no cycles**.

Run the cell below to build a Precedence Graph analyzer and test two schedules:
* `schedule_1`: $R_1(A), W_1(A), R_2(A), W_2(A), R_1(B), W_1(B), R_2(B), W_2(B)$
* `schedule_2`: $R_1(A), R_2(A), W_1(A), W_2(A)$


In [ ]:
def is_conflict_serializable(schedule):
    """
    Takes a list of operations formatted as (Tx_ID, 'R' or 'W', Data_Item)
    Builds the Precedence Graph and checks if it is acyclic (Conflict Serializable).
    """
    edges = set()
    txs = {op[0] for op in schedule}
    
    for i in range(len(schedule)):
        t1, op1, item1 = schedule[i]
        for j in range(i + 1, len(schedule)):
            t2, op2, item2 = schedule[j]
            if t1 != t2 and item1 == item2 and ('W' in (op1, op2)):
                edges.add((t1, t2))
                
    # Detect cycle using DFS
    adj = {t: [] for t in txs}
    for u, v in edges:
        adj[u].append(v)
        
    visited = {} # 0: unvisited, 1: visiting, 2: visited
    def has_cycle(node):
        visited[node] = 1
        for nei in adj[node]:
            if visited.get(nei, 0) == 1:
                return True
            if visited.get(nei, 0) == 0 and has_cycle(nei):
                return True
        visited[node] = 2
        return False
        
    cyclic = any(has_cycle(t) for t in txs if visited.get(t, 0) == 0)
    return (not cyclic), edges

schedule_1 = [('T1','R','A'), ('T1','W','A'), ('T2','R','A'), ('T2','W','A'),
              ('T1','R','B'), ('T1','W','B'), ('T2','R','B'), ('T2','W','B')]

schedule_2 = [('T1','R','A'), ('T2','R','A'), ('T1','W','A'), ('T2','W','A')]

cs1, edges1 = is_conflict_serializable(schedule_1)
cs2, edges2 = is_conflict_serializable(schedule_2)

print(f"Schedule 1 -> Precedence Edges: {edges1} | Conflict Serializable? {cs1}")
print(f"Schedule 2 -> Precedence Edges: {edges2} | Conflict Serializable? {cs2}")
assert cs1 is True and cs2 is False
print("Task 5.2 Conflict Serializability Verification Passed!")


### Task 5.3: `B+ Tree` Indexing & `EXPLAIN QUERY PLAN` Performance Benchmark
1. We generate **20,000 synthetic student records** in `Large_Students_Table`.
2. We inspect `EXPLAIN QUERY PLAN` **before** creating an index (`SCAN Large_Students_Table` — Full Table Scan).
3. We create a `B+ Tree` Secondary Index `idx_student_email` on `email` and a range index `idx_student_gpa` on `gpa`.
4. We inspect `EXPLAIN QUERY PLAN` **after** indexing (`SEARCH Large_Students_Table USING INDEX`) and benchmark the query speedup.


In [ ]:
# 1. Create and populate 20,000 rows in Large_Students_Table
exec_script("""
DROP TABLE IF EXISTS Large_Students_Table;
CREATE TABLE Large_Students_Table (
    student_id INTEGER PRIMARY KEY,
    email TEXT NOT NULL,
    dept_code TEXT NOT NULL,
    gpa REAL NOT NULL
);
""")

bulk_rows = [(i, f"student_{i}@univ.edu", ("CS" if i % 3 == 0 else "EE"), round((i % 400) / 100.0, 2)) for i in range(1, 20001)]
conn.executemany("INSERT INTO Large_Students_Table VALUES (?, ?, ?, ?);", bulk_rows)
conn.commit()

# 2. Check Query Plan BEFORE B+ Tree Index
plan_before = run_query("EXPLAIN QUERY PLAN SELECT * FROM Large_Students_Table WHERE email = 'student_19500@univ.edu';")
print("Query Plan BEFORE Index:")
display(plan_before)

t0 = time.perf_counter()
for _ in range(100):
    conn.execute("SELECT * FROM Large_Students_Table WHERE email = 'student_19500@univ.edu';").fetchall()
time_no_index = (time.perf_counter() - t0) * 1000

# 3. Create B+ Tree Secondary Index on email
exec_sql("CREATE INDEX idx_student_email ON Large_Students_Table(email);")

# 4. Check Query Plan AFTER B+ Tree Index
plan_after = run_query("EXPLAIN QUERY PLAN SELECT * FROM Large_Students_Table WHERE email = 'student_19500@univ.edu';")
print("Query Plan AFTER B+ Tree Index:")
display(plan_after)

t1 = time.perf_counter()
for _ in range(100):
    conn.execute("SELECT * FROM Large_Students_Table WHERE email = 'student_19500@univ.edu';").fetchall()
time_with_index = (time.perf_counter() - t1) * 1000

print(f"100 Lookups WITHOUT Index (Full Scan): {time_no_index:.2f} ms")
print(f"100 Lookups WITH B+ Tree Index:        {time_with_index:.2f} ms (Speedup: {time_no_index / max(time_with_index, 0.01):.1f}x)")
assert "USING INDEX idx_student_email" in plan_after.iloc[0]["detail"], "Expected B+ Tree index usage in query plan!"
print("Task 5.3 B+ Tree Indexing Lab Passed!")
